In [ ]:
# ── Install packages not pre-loaded in Colab
!pip install datasets tiktoken -q

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU memory: {total_mem:.1f} GB")
    print("✓ Good to go.")
else:
    print("⚠ No GPU. Go to Runtime → Change runtime type → T4 GPU, then re-run.")

Device: cuda
GPU: Tesla T4
Total GPU memory: 15.6 GB
✓ Good to go.


In [ ]:
import math, time, json, os
from dataclasses import dataclass
from typing import Optional
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tiktoken

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────────
# ModelConfig — the SINGLE place we change to swap components
# Every experiment is just: cfg = ModelConfig(attention_type='linear')
# Everything else — training loop, data, logging — stays identical
# ─────────────────────────────────────────────────────────────────
@dataclass
class ModelConfig:

    # Model size
    vocab_size   : int   = 50257   # GPT-2 tokenizer vocabulary
    d_model      : int   = 256     # width of every vector in the model
    n_heads      : int   = 4       # how many attention heads per layer
    n_layers     : int   = 4       # how many transformer blocks stacked
    d_ff         : int   = 1024    # hidden size inside feed-forward (= 4 × d_model)
    dropout      : float = 0.1
    max_seq_len  : int   = 512     # context window length during training

    # Attention variant — swap this to change Part 2
    # 'standard'       → full softmax attention  (baseline)
    # 'sliding_window' → each token only sees ±window_size neighbours
    # 'linear'         → kernel trick, O(n) instead of O(n²)
    # 'gqa'            → grouped query attention, fewer KV heads
    attention_type : str = 'standard'
    window_size    : int = 64   # used only when attention_type='sliding_window'
    n_kv_heads     : int = 2    # used only when attention_type='gqa'

    # Positional encoding variant — swap this to change Part 3
    # 'sinusoidal' → fixed sine/cosine waves (baseline)
    # 'rope'       → rotate Q and K by position angle
    # 'alibi'      → subtract distance penalty from attention scores
    # 'relative'   → learned bias per relative distance
    pe_type : str = 'sinusoidal'

    # Convolution hybrid — swap this to change Part 4
    # None               → no conv (baseline)
    # 'conv_before'      → Conv1D before every attention block
    # 'gated_conv_ffn'   → replace feed-forward MLP with gated conv
    conv_type        : Optional[str] = None
    conv_kernel_size : int = 3

# Print the default config so we can see exactly what we're starting with
cfg = ModelConfig()
print("ModelConfig (default = baseline):")
print(f"  vocab_size    : {cfg.vocab_size}")
print(f"  d_model       : {cfg.d_model}")
print(f"  n_heads       : {cfg.n_heads}")
print(f"  n_layers      : {cfg.n_layers}")
print(f"  d_ff          : {cfg.d_ff}")
print(f"  max_seq_len   : {cfg.max_seq_len}")
print(f"  attention_type: {cfg.attention_type}")
print(f"  pe_type       : {cfg.pe_type}")
print(f"  conv_type     : {cfg.conv_type}")
print("\nAll imports OK ✓")


ModelConfig (default = baseline):
  vocab_size    : 50257
  d_model       : 256
  n_heads       : 4
  n_layers      : 4
  d_ff          : 1024
  max_seq_len   : 512
  attention_type: standard
  pe_type       : sinusoidal
  conv_type     : None

All imports OK ✓


In [ ]:
print("Downloading WikiText-2...")
raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")  # ← fixed
print(f"  Train articles : {len(raw['train']):,}")
print(f"  Val articles   : {len(raw['validation']):,}")

enc = tiktoken.get_encoding("gpt2")
print(f"\nTokenizer: GPT-2  |  Vocab size: {enc.n_vocab:,}")

def tokenize_split(split):
    all_text = "\n\n".join(x for x in raw[split]["text"] if x.strip())
    tokens   = enc.encode(all_text)
    print(f"  {split:12s}: {len(tokens):>10,} tokens  ({len(tokens)/1e6:.2f}M)")
    return tokens

print("\nTokenizing:")
train_tokens = tokenize_split("train")
val_tokens   = tokenize_split("validation")

class TokenDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.seq_len = seq_len
        n_chunks     = (len(tokens) - 1) // seq_len
        self.data    = torch.tensor(tokens[:n_chunks * seq_len + 1], dtype=torch.long)

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.data[start     : start + self.seq_len]
        y = self.data[start + 1 : start + self.seq_len + 1]
        return x, y

def make_loaders(seq_len, batch_size):
    train_ds = TokenDataset(train_tokens, seq_len)
    val_ds   = TokenDataset(val_tokens,   seq_len)
    train_dl = DataLoader(train_ds, batch_size=batch_size,
                          shuffle=True,  pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size,
                          shuffle=False, pin_memory=True, drop_last=True)
    return train_dl, val_dl

train_dl, val_dl = make_loaders(seq_len=512, batch_size=16)

print(f"\nDataLoaders (seq_len=512, batch_size=16):")
print(f"  Train batches : {len(train_dl):,}")
print(f"  Val batches   : {len(val_dl):,}")

x_sample, y_sample = next(iter(train_dl))
print(f"\nSample shapes : x={tuple(x_sample.shape)}, y={tuple(y_sample.shape)}")
print(f"First 10 token IDs : {x_sample[0, :10].tolist()}")
print(f"Decoded            : '{enc.decode(x_sample[0, :10].tolist())}'")
print("\nData pipeline OK ✓")

  Train articles : 36,718
  Val articles   : 3,760

Tokenizer: GPT-2  |  Vocab size: 50,257

Tokenizing:
  train       :  2,415,650 tokens  (2.42M)
  validation  :    249,749 tokens  (0.25M)

DataLoaders (seq_len=512, batch_size=16):
  Train batches : 294
  Val batches   : 30

Sample shapes : x=(16, 512), y=(16, 512)
First 10 token IDs : [262, 2347, 7619, 2318, 1719, 12776, 7087, 3481, 41882, 509]
Decoded            : ' the mass tort bar having vociferously discredited K'

Data pipeline OK ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — Full Modular Transformer Architecture
# All variants controlled by ModelConfig fields
# ═══════════════════════════════════════════════════════════════════

torch.manual_seed(42)

# ───────────────────────────────────────────────────────────────────
# PART A: Positional Encoding Helpers
# ───────────────────────────────────────────────────────────────────

def make_sinusoidal_pe(max_len: int, d_model: int) -> torch.Tensor:
    """Fixed sine/cosine positional encoding. Shape: (1, max_len, d_model)"""
    pe  = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() *
                    (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe.unsqueeze(0)                          # (1, T, d_model)


def make_rope_cache(max_len: int, d_head: int, dev) -> tuple:
    """
    Precompute RoPE rotation angles.
    Returns cos, sin each of shape (1, 1, max_len, d_head).
    """
    theta = 1.0 / (10000.0 ** (
        torch.arange(0, d_head, 2, device=dev).float() / d_head))
    pos   = torch.arange(max_len, device=dev).float()
    freqs = torch.outer(pos, theta)                 # (T, d_head//2)
    freqs = torch.cat([freqs, freqs], dim=-1)       # (T, d_head)
    return freqs.cos()[None, None], freqs.sin()[None, None]


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Rotate vector: [-x_back_half, x_front_half]"""
    h = x.shape[-1] // 2
    return torch.cat([-x[..., h:], x[..., :h]], dim=-1)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    """Apply RoPE rotation to Q or K. x: (B, H, T, d_head)"""
    T = x.shape[2]
    return x * cos[:, :, :T] + rotate_half(x) * sin[:, :, :T]


def make_alibi_bias(n_heads: int, max_len: int, dev) -> torch.Tensor:
    """
    ALiBi bias matrix. Shape: (1, n_heads, max_len, max_len).
    Past/current positions: -slope * distance.
    Future positions: -inf (causal mask built in).
    """
    exp    = 8.0 / n_heads
    slopes = torch.tensor(
        [2.0 ** (-exp * h) for h in range(1, n_heads + 1)], device=dev)
    pos    = torch.arange(max_len, device=dev)
    dist   = pos.unsqueeze(1) - pos.unsqueeze(0)        # (T, T), positive=past
    bias   = -slopes.view(-1, 1, 1) * dist.abs().unsqueeze(0)   # (H, T, T)
    bias   = bias.masked_fill(dist.unsqueeze(0) < 0, float('-inf'))
    return bias.unsqueeze(0)                            # (1, H, T, T)


# ───────────────────────────────────────────────────────────────────
# PART B: Attention Module
# ───────────────────────────────────────────────────────────────────

class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg    = cfg
        self.n_h    = cfg.n_heads
        self.d_h    = cfg.d_model // cfg.n_heads
        self.atype  = cfg.attention_type
        # GQA uses fewer KV heads; everything else uses same count as Q
        self.n_kv   = cfg.n_kv_heads if self.atype == 'gqa' else cfg.n_heads
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"

        self.q_proj   = nn.Linear(cfg.d_model, self.n_h  * self.d_h, bias=False)
        self.k_proj   = nn.Linear(cfg.d_model, self.n_kv * self.d_h, bias=False)
        self.v_proj   = nn.Linear(cfg.d_model, self.n_kv * self.d_h, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.drop     = nn.Dropout(cfg.dropout)

        # Pre-compute and register buffers for PE methods that live inside attention
        if cfg.pe_type == 'rope':
            c, s = make_rope_cache(cfg.max_seq_len, self.d_h, 'cpu')
            self.register_buffer('rope_cos', c)
            self.register_buffer('rope_sin', s)

        if cfg.pe_type == 'alibi':
            self.register_buffer('alibi_bias',
                make_alibi_bias(cfg.n_heads, cfg.max_seq_len, 'cpu'))

        if cfg.pe_type == 'relative':
            # Learnable bias: one scalar per (relative_distance, head)
            self.rel_bias = nn.Embedding(cfg.max_seq_len, cfg.n_heads)

    # ── Main forward ────────────────────────────────────────────────
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.n_h,  self.d_h).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_h).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_h).transpose(1, 2)
        # Q: (B, n_heads, T, d_head)
        # K,V: (B, n_kv, T, d_head)  ← may be fewer heads in GQA

        # GQA: expand K,V to match n_heads by repeating
        if self.atype == 'gqa':
            rep = self.n_h // self.n_kv
            k   = k.repeat_interleave(rep, dim=1)
            v   = v.repeat_interleave(rep, dim=1)

        # RoPE: rotate Q and K BEFORE computing scores
        if self.cfg.pe_type == 'rope':
            q = apply_rope(q, self.rope_cos, self.rope_sin)
            k = apply_rope(k, self.rope_cos, self.rope_sin)

        # ── Linear attention takes a completely different path ──────
        if self.atype == 'linear':
            out = self._linear_attn(q, k, v)               # (B, H, T, d_h)
            return self.out_proj(
                out.transpose(1, 2).contiguous().view(B, T, -1))

        # ── Score matrix for standard / sliding_window / gqa ───────
        scale  = self.d_h ** -0.5
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, H, T, T)

        # Add positional bias / causal mask (depends on pe_type)
        if self.cfg.pe_type == 'alibi':
            # ALiBi bias already contains causal -inf for future positions
            scores = scores + self.alibi_bias[:, :, :T, :T]

        elif self.cfg.pe_type == 'relative':
            # Relative PE bias + causal mask, computed on-the-fly
            scores = scores + self._relative_bias(T, x.device)

        else:
            # Sinusoidal and RoPE: add standard causal mask
            causal = torch.triu(
                torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(
                causal.unsqueeze(0).unsqueeze(0), float('-inf'))

        # Sliding window: additionally block tokens outside the window
        if self.atype == 'sliding_window':
            i = torch.arange(T, device=x.device)
            outside = (i.unsqueeze(1) - i.unsqueeze(0)) > self.cfg.window_size
            scores = scores.masked_fill(
                outside.unsqueeze(0).unsqueeze(0), float('-inf'))

        attn = self.drop(F.softmax(scores, dim=-1))
        out  = torch.matmul(attn, v)                        # (B, H, T, d_h)
        return self.out_proj(
            out.transpose(1, 2).contiguous().view(B, T, -1))

    # ── Linear attention helper ─────────────────────────────────────
    def _linear_attn(self, q, k, v) -> torch.Tensor:
        """
        Causal linear attention via ELU+1 kernel + cumulative sum.
        Avoids computing the full T×T attention matrix (O(n²) → O(n)).
        φ(x) = ELU(x) + 1  keeps all values positive (required for stability).
        """
        phi_q = F.elu(q) + 1.0    # (B, H, T, d_head)
        phi_k = F.elu(k) + 1.0

        # kv[t] = outer(phi_k[t], v[t])  →  shape (B, H, T, d_head, d_head)
        kv = torch.einsum('bhtk,bhtv->bhtkv', phi_k, v)
        # Causal cumsum: S[t] = sum_{s<=t} kv[s]
        S  = torch.cumsum(kv, dim=2)
        # Normalizer: z[t] = sum_{s<=t} phi_k[s]
        z  = torch.cumsum(phi_k, dim=2)

        out  = torch.einsum('bhtk,bhtkv->bhtv', phi_q, S)
        norm = (phi_q * z).sum(dim=-1, keepdim=True).clamp(min=1e-6)
        return out / norm

    # ── Relative PE bias helper ─────────────────────────────────────
    def _relative_bias(self, T: int, dev) -> torch.Tensor:
        """
        Compute learned relative position bias table lookup + causal mask.
        dist[i,j] = i - j (clamped to [0, max_seq_len-1] for past positions).
        Returns (1, n_heads, T, T).
        """
        idx  = torch.arange(T, device=dev)
        dist = (idx.unsqueeze(1) - idx.unsqueeze(0)).clamp(
            0, self.cfg.max_seq_len - 1)               # (T, T)
        bias = self.rel_bias(dist).permute(2, 0, 1)   # (n_heads, T, T)
        causal = torch.triu(
            torch.full((T, T), float('-inf'), device=dev), diagonal=1)
        return (bias + causal).unsqueeze(0)            # (1, H, T, T)


# ───────────────────────────────────────────────────────────────────
# PART C: Feed-Forward Network
# ───────────────────────────────────────────────────────────────────

class FFN(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg  = cfg
        pad       = cfg.conv_kernel_size // 2

        if cfg.conv_type == 'gated_conv_ffn':
            # Two conv branches: value branch (what to output) and gate (how much)
            self.conv_v = nn.Conv1d(cfg.d_model, cfg.d_ff,
                                    cfg.conv_kernel_size, padding=pad)
            self.conv_g = nn.Conv1d(cfg.d_model, cfg.d_ff,
                                    cfg.conv_kernel_size, padding=pad)
            self.proj   = nn.Linear(cfg.d_ff, cfg.d_model)
        else:
            # Standard MLP: Linear → GELU → Linear
            self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff)
            self.fc2 = nn.Linear(cfg.d_ff,    cfg.d_model)

        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.cfg.conv_type == 'gated_conv_ffn':
            xt   = x.transpose(1, 2)                          # (B, d_model, T)
            val  = self.conv_v(xt).transpose(1, 2)            # (B, T, d_ff)
            gate = torch.sigmoid(self.conv_g(xt).transpose(1, 2))
            return self.drop(self.proj(val * gate))           # gated output

        return self.drop(self.fc2(F.gelu(self.fc1(x))))       # standard MLP


# ───────────────────────────────────────────────────────────────────
# PART D: Transformer Block
# ───────────────────────────────────────────────────────────────────

class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.ffn  = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

        # Optional depthwise conv before attention (conv_before variant)
        if cfg.conv_type == 'conv_before':
            pad = cfg.conv_kernel_size // 2
            self.conv_pre = nn.Sequential(
                nn.Conv1d(cfg.d_model, cfg.d_model,
                          cfg.conv_kernel_size, padding=pad,
                          groups=cfg.d_model),   # depthwise = one filter per channel
                nn.GELU()
            )
        else:
            self.conv_pre = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Optional local feature extraction before attention
        if self.conv_pre is not None:
            x = x + self.conv_pre(x.transpose(1, 2)).transpose(1, 2)

        # Attention sub-layer with pre-LayerNorm and residual
        x = x + self.drop(self.attn(self.ln1(x)))

        # FFN sub-layer with pre-LayerNorm and residual
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


# ───────────────────────────────────────────────────────────────────
# PART E: Full Transformer Language Model
# ───────────────────────────────────────────────────────────────────

class TransformerLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg     = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        # Sinusoidal PE is the only type added at the embedding level
        # RoPE, ALiBi, and Relative PE are handled inside each Attention module
        if cfg.pe_type == 'sinusoidal':
            self.register_buffer(
                'sin_pe', make_sinusoidal_pe(cfg.max_seq_len, cfg.d_model))

        self.emb_drop = nn.Dropout(cfg.dropout)
        self.blocks   = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f     = nn.LayerNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Init weights first, THEN tie lm_head weights to tok_emb
        # (saves ~13M parameters — standard practice for LMs)
        self.apply(self._init_weights)
        self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.tok_emb(idx)                      # (B, T, d_model)
        if self.cfg.pe_type == 'sinusoidal':
            x = x + self.sin_pe[:, :T, :]
        x = self.emb_drop(x)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))          # (B, T, vocab_size)

    def num_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ───────────────────────────────────────────────────────────────────
# SANITY CHECK: Forward pass through ALL 9 variants
# ───────────────────────────────────────────────────────────────────

test_configs = [
    ("Baseline (standard+sinusoidal)", ModelConfig()),
    ("Sliding Window Attention",        ModelConfig(attention_type='sliding_window')),
    ("Linear Attention",                ModelConfig(attention_type='linear')),
    ("GQA (n_kv_heads=2)",              ModelConfig(attention_type='gqa')),
    ("RoPE",                            ModelConfig(pe_type='rope')),
    ("ALiBi",                           ModelConfig(pe_type='alibi')),
    ("Relative PE",                     ModelConfig(pe_type='relative')),
    ("Conv-Before-Attention",           ModelConfig(conv_type='conv_before')),
    ("Gated Conv FFN",                  ModelConfig(conv_type='gated_conv_ffn')),
]

dummy = torch.randint(0, 50257, (2, 64)).to(device)

print(f"{'Config':<35} {'Params':>12}  Status")
print("─" * 60)

all_ok = True
for name, cfg in test_configs:
    try:
        m = TransformerLM(cfg).to(device)
        with torch.no_grad():
            out = m(dummy)
        finite = torch.isfinite(out).all().item()
        status = "✓ OK" if finite else "⚠ NaN in output"
        if not finite:
            all_ok = False
        print(f"{name:<35} {m.num_params():>12,}  {status}")
        del m
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"{name:<35} {'---':>12}  ✗ ERROR: {e}")
        all_ok = False

print("─" * 60)
print("All variants OK ✓" if all_ok else "⚠ Some variants failed — share output above")

Config                                    Params  Status
────────────────────────────────────────────────────────────
Baseline (standard+sinusoidal)        16,021,248  ✓ OK
Sliding Window Attention              16,021,248  ✓ OK
Linear Attention                      16,021,248  ✓ OK
GQA (n_kv_heads=2)                    15,759,104  ✓ OK
RoPE                                  16,021,248  ✓ OK
ALiBi                                 16,021,248  ✓ OK
Relative PE                           16,029,440  ✓ OK
Conv-Before-Attention                 16,025,344  ✓ OK
Gated Conv FFN                        21,268,224  ✓ OK
────────────────────────────────────────────────────────────
All variants OK ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 — Training Infrastructure
# train_model()  : trains one config, returns results dict
# evaluate()     : computes val perplexity
# All experiments use this exact same code — fair comparison
# ═══════════════════════════════════════════════════════════════════

# Master results store — all experiments accumulate here
all_results = {}

def get_gpu_mem_mb() -> float:
    """Peak GPU memory allocated since last reset, in MB."""
    return torch.cuda.max_memory_allocated() / 1e6

def evaluate(model: nn.Module, val_dl: DataLoader) -> tuple:
    """
    Run one pass over the validation set.
    Returns (avg_loss, perplexity, tokens_per_sec).
    Perplexity = exp(avg_cross_entropy_loss).
    """
    model.eval()
    total_loss, total_batches = 0.0, 0
    t0 = time.time()
    total_tokens = 0

    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            logits = model(x)                          # (B, T, vocab_size)
            loss   = F.cross_entropy(
                logits.view(-1, logits.size(-1)),      # (B*T, vocab_size)
                y.view(-1)                             # (B*T,)
            )
            total_loss    += loss.item()
            total_batches += 1
            total_tokens  += x.numel()

    elapsed     = time.time() - t0
    avg_loss    = total_loss / total_batches
    perplexity  = math.exp(avg_loss)
    tok_per_sec = total_tokens / elapsed
    return avg_loss, perplexity, tok_per_sec


def train_model(
    name      : str,
    cfg       : ModelConfig,
    n_epochs  : int = 3,
    seq_len   : int = 512,
    batch_size: int = 16,
    lr        : float = 3e-4,
    verbose   : bool = True
) -> dict:
    """
    Train one model config end-to-end, log all metrics.

    Returns a dict with:
      - train_losses, val_losses, val_perplexities  per epoch
      - peak_gpu_mem_mb
      - avg_epoch_time_sec
      - val_tok_per_sec  (inference throughput)
    """
    if verbose:
        print(f"\n{'═'*60}")
        print(f"  Experiment: {name}")
        print(f"  attention={cfg.attention_type}  pe={cfg.pe_type}"
              f"  conv={cfg.conv_type}  seq_len={seq_len}")
        print(f"{'═'*60}")

    # Fresh data loaders for this config's seq_len
    tr_dl, vl_dl = make_loaders(seq_len, batch_size)

    # Build model
    torch.cuda.reset_peak_memory_stats()
    model = TransformerLM(cfg).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.1,
        betas=(0.9, 0.95), eps=1e-8
    )

    # Linear warmup (10% of total steps) then cosine decay
    total_steps   = n_epochs * len(tr_dl)
    warmup_steps  = total_steps // 10

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # ── Training ──────────────────────────────────────────────────
    train_losses, val_losses, val_ppls = [], [], []
    epoch_times = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss   = 0.0
        epoch_tokens = 0
        t_epoch      = time.time()

        for step, (x, y) in enumerate(tr_dl, 1):
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss   = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss   += loss.item()
            epoch_tokens += x.numel()

            # Print progress every 100 steps
            if verbose and step % 100 == 0:
                avg = epoch_loss / step
                ppl = math.exp(avg)
                print(f"  Epoch {epoch} | step {step:>4}/{len(tr_dl)}"
                      f" | loss {avg:.4f} | ppl {ppl:.1f}")

        # ── End of epoch ──────────────────────────────────────────
        elapsed     = time.time() - t_epoch
        avg_tr_loss = epoch_loss / len(tr_dl)
        val_loss, val_ppl, val_tps = evaluate(model, vl_dl)

        train_losses.append(avg_tr_loss)
        val_losses  .append(val_loss)
        val_ppls    .append(val_ppl)
        epoch_times .append(elapsed)

        peak_mem = get_gpu_mem_mb()

        if verbose:
            print(f"\n  ── Epoch {epoch}/{n_epochs} summary ──")
            print(f"     train loss : {avg_tr_loss:.4f}")
            print(f"     val loss   : {val_loss:.4f}")
            print(f"     val ppl    : {val_ppl:.2f}")
            print(f"     epoch time : {elapsed:.0f}s")
            print(f"     peak mem   : {peak_mem:.0f} MB")
            print(f"     val tps    : {val_tps:.0f} tok/s\n")

    results = {
        "name"              : name,
        "attention_type"    : cfg.attention_type,
        "pe_type"           : cfg.pe_type,
        "conv_type"         : str(cfg.conv_type),
        "seq_len"           : seq_len,
        "n_params"          : model.num_params(),
        "train_losses"      : train_losses,
        "val_losses"        : val_losses,
        "val_perplexities"  : val_ppls,
        "best_val_ppl"      : min(val_ppls),
        "final_val_ppl"     : val_ppls[-1],
        "avg_epoch_time_sec": sum(epoch_times) / len(epoch_times),
        "peak_gpu_mem_mb"   : peak_mem,
        "val_tok_per_sec"   : val_tps,
    }

    all_results[name] = results

    if verbose:
        print(f"  ✓ Done — best val ppl: {min(val_ppls):.2f}")

    del model
    torch.cuda.empty_cache()
    return results


def print_results_table():
    """Print a clean comparison table of all completed experiments."""
    if not all_results:
        print("No experiments run yet.")
        return

    print(f"\n{'─'*95}")
    print(f"{'Experiment':<32} {'Attn':<16} {'PE':<12} {'Conv':<16}"
          f" {'Best PPL':>9} {'Mem(MB)':>8} {'s/epoch':>8} {'tok/s':>8}")
    print(f"{'─'*95}")

    for r in all_results.values():
        print(
            f"{r['name']:<32} "
            f"{r['attention_type']:<16} "
            f"{r['pe_type']:<12} "
            f"{r['conv_type']:<16} "
            f"{r['best_val_ppl']:>9.2f} "
            f"{r['peak_gpu_mem_mb']:>8.0f} "
            f"{r['avg_epoch_time_sec']:>8.0f} "
            f"{r['val_tok_per_sec']:>8.0f}"
        )

    print(f"{'─'*95}")


print("Training infrastructure ready.")
print(f"  all_results dict  : {type(all_results)}")
print(f"  evaluate()        : ✓")
print(f"  train_model()     : ✓")
print(f"  print_results_table() : ✓")
print("\nReady to run experiments.")

Training infrastructure ready.
  all_results dict  : <class 'dict'>
  evaluate()        : ✓
  train_model()     : ✓
  print_results_table() : ✓

Ready to run experiments.


In [ ]:
# Reset results store
all_results = {}

# Re-run baseline with 10 epochs
baseline_cfg = ModelConfig(
    attention_type='standard',
    pe_type='sinusoidal',
    conv_type=None,
    max_seq_len=1024
)

train_model(
    name       = "Baseline",
    cfg        = baseline_cfg,
    n_epochs   = 10,
    seq_len    = 512,
    batch_size = 16,
    lr         = 3e-4,
    verbose    = True
)

print_results_table()


════════════════════════════════════════════════════════════
  Experiment: Baseline
  attention=standard  pe=sinusoidal  conv=None  seq_len=512
════════════════════════════════════════════════════════════
  Epoch 1 | step  100/294 | loss 10.2594 | ppl 28550.2
  Epoch 1 | step  200/294 | loss 9.2066 | ppl 9962.6

  ── Epoch 1/10 summary ──
     train loss : 8.6427
     val loss   : 7.4844
     val ppl    : 1780.04
     epoch time : 92s
     peak mem   : 8013 MB
     val tps    : 71608 tok/s

  Epoch 2 | step  100/294 | loss 7.4421 | ppl 1706.3
  Epoch 2 | step  200/294 | loss 7.4456 | ppl 1712.3

  ── Epoch 2/10 summary ──
     train loss : 7.4396
     val loss   : 7.3742
     val ppl    : 1594.30
     epoch time : 103s
     peak mem   : 8013 MB
     val tps    : 66067 tok/s

  Epoch 3 | step  100/294 | loss 7.2557 | ppl 1416.2
  Epoch 3 | step  200/294 | loss 7.1833 | ppl 1317.2

  ── Epoch 3/10 summary ──
     train loss : 7.1176
     val loss   : 6.9415
     val ppl    : 1034.29
   

In [ ]:
# ═══════════════════════════════════════════════
# EXPERIMENTS 2-4: Attention Variants
# Same sinusoidal PE, same training — only attention_type changes
# After training all 3, we also evaluate each at seq_len=1024
# to measure context-length behaviour
# ═══════════════════════════════════════════════

attn_cfgs = [
    ("Attn: Sliding Window", ModelConfig(attention_type='sliding_window',
                                         max_seq_len=1024, window_size=64)),
    ("Attn: Linear",         ModelConfig(attention_type='linear',
                                         max_seq_len=1024)),
    ("Attn: GQA",            ModelConfig(attention_type='gqa',
                                         max_seq_len=1024, n_kv_heads=2)),
]

for name, cfg in attn_cfgs:
    train_model(name=name, cfg=cfg,
                n_epochs=10, seq_len=512, batch_size=16,
                lr=3e-4, verbose=True)

print_results_table()

# ── Context-length sweep ────────────────────────────────────────────
# Re-build each trained config at seq_len=1024 and evaluate perplexity
# Training was at 512; this tests how each attention type handles longer input
print("\n" + "═"*60)
print("CONTEXT LENGTH SWEEP  (model trained on 512, eval at 512 & 1024)")
print("─"*60)
print(f"{'Config':<28} {'PPL@512':>10} {'PPL@1024':>10} {'Δ PPL':>10}")
print("─"*60)

sweep_names = ["Baseline"] + [n for n, _ in attn_cfgs]
sweep_cfgs  = [ModelConfig(max_seq_len=1024)] + [c for _, c in attn_cfgs]

for name, cfg in zip(sweep_names, sweep_cfgs):
    # Rebuild model with same config and SAME random init for fair comparison
    torch.manual_seed(42)
    torch.cuda.reset_peak_memory_stats()
    m = TransformerLM(cfg).to(device)

    # Re-train quickly for 3 epochs (same as stored results)
    tr_dl_512, vl_dl_512  = make_loaders(512,  16)
    _,         vl_dl_1024 = make_loaders(1024, 8)

    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.1,
                             betas=(0.9, 0.95))
    steps = 3 * len(tr_dl_512)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps, eta_min=3e-5)

    m.train()
    for x, y in tr_dl_512:
        x, y = x.to(device), y.to(device)
        logits = m(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step(); sched.step()

    m.eval()
    _, ppl_512,  _ = evaluate(m, vl_dl_512)
    _, ppl_1024, _ = evaluate(m, vl_dl_1024)
    delta = ppl_1024 - ppl_512

    print(f"{name:<28} {ppl_512:>10.2f} {ppl_1024:>10.2f} {delta:>+10.2f}")
    del m; torch.cuda.empty_cache()

print("═"*60)
print("\n(Δ PPL = how much worse at 1024 vs 512. Smaller = better extrapolation.)")


════════════════════════════════════════════════════════════
  Experiment: Attn: Sliding Window
  attention=sliding_window  pe=sinusoidal  conv=None  seq_len=512
════════════════════════════════════════════════════════════
  Epoch 1 | step  100/294 | loss 10.2493 | ppl 28262.8
  Epoch 1 | step  200/294 | loss 9.1991 | ppl 9888.5

  ── Epoch 1/10 summary ──
     train loss : 8.6357
     val loss   : 7.4728
     val ppl    : 1759.46
     epoch time : 106s
     peak mem   : 8016 MB
     val tps    : 64466 tok/s

  Epoch 2 | step  100/294 | loss 7.4357 | ppl 1695.4
  Epoch 2 | step  200/294 | loss 7.4454 | ppl 1712.0

  ── Epoch 2/10 summary ──
     train loss : 7.4464
     val loss   : 7.4939
     val ppl    : 1797.12
     epoch time : 106s
     peak mem   : 8016 MB
     val tps    : 64638 tok/s

  Epoch 3 | step  100/294 | loss 7.3906 | ppl 1620.7
  Epoch 3 | step  200/294 | loss 7.3019 | ppl 1483.0

  ── Epoch 3/10 summary ──
     train loss : 7.2184
     val loss   : 6.9855
     val p

In [ ]:
# ═══════════════════════════════════════════════
# EXPERIMENTS 5-8: Positional Encoding Variants
# All trained on seq_len=512, then evaluated at 512, 1024, 2048
# max_seq_len=2048 so PE buffers cover the full eval range
# This is the extrapolation test from Part 3 of the assignment
# ═══════════════════════════════════════════════

extrap_results = {}

def train_and_extrapolate(pe_name: str):
    """
    Train a model with given PE type on seq_len=512.
    Then evaluate on seq_len=512, 1024, 2048 WITHOUT retraining.
    Returns the model for caller to delete.
    """
    print(f"\n{'═'*55}")
    print(f"  PE Experiment: {pe_name.upper()}")
    print(f"{'═'*55}")

    cfg     = ModelConfig(pe_type=pe_name, max_seq_len=2048)
    tr_dl,_ = make_loaders(512, 16)

    torch.cuda.reset_peak_memory_stats()
    model = TransformerLM(cfg).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=3e-4,
                               weight_decay=0.1, betas=(0.9, 0.95))
    total_steps  = 3 * len(tr_dl)
    warmup_steps = total_steps // 10

    def lr_fn(s):
        if s < warmup_steps:
            return s / max(1, warmup_steps)
        p = (s - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

    # ── Train 3 epochs ──────────────────────────────────────────────
    val_ppls_512 = []
    epoch_times  = []
    _, vl_512 = make_loaders(512, 16)

    for epoch in range(1, 11):
        model.train()
        ep_loss = 0.0
        t0 = time.time()

        for step, (x, y) in enumerate(tr_dl, 1):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = F.cross_entropy(
                logits.view(-1, logits.size(-1)), y.view(-1))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            ep_loss += loss.item()

            if step % 100 == 0:
                print(f"  Epoch {epoch} | step {step:>3}/{len(tr_dl)}"
                      f" | loss {ep_loss/step:.4f}")

        elapsed = time.time() - t0
        epoch_times.append(elapsed)

        _, vl_ppl, vl_tps = evaluate(model, vl_512)
        val_ppls_512.append(vl_ppl)
        print(f"  ── Epoch {epoch} | val_ppl@512={vl_ppl:.2f}"
              f" | time={elapsed:.0f}s\n")

    # Store in main results table
    all_results[f"PE: {pe_name}"] = {
        "name"              : f"PE: {pe_name}",
        "attention_type"    : "standard",
        "pe_type"           : pe_name,
        "conv_type"         : "None",
        "seq_len"           : 512,
        "n_params"          : model.num_params(),
        "best_val_ppl"      : min(val_ppls_512),
        "final_val_ppl"     : val_ppls_512[-1],
        "avg_epoch_time_sec": sum(epoch_times) / len(epoch_times),
        "peak_gpu_mem_mb"   : get_gpu_mem_mb(),
        "val_tok_per_sec"   : vl_tps,
        "val_perplexities"  : val_ppls_512,
    }

    # ── Extrapolation eval ──────────────────────────────────────────
    print(f"  Extrapolation test for {pe_name}:")
    row = {"pe_type": pe_name}

    for eval_len, bs in [(512, 16), (1024, 8), (2048, 4)]:
        try:
            _, vl_eval = make_loaders(eval_len, bs)
            _, ppl, _  = evaluate(model, vl_eval)
            row[f"ppl_{eval_len}"] = round(ppl, 2)
            print(f"    seq_len={eval_len:>4}: ppl = {ppl:.2f}")
        except Exception as e:
            row[f"ppl_{eval_len}"] = None
            print(f"    seq_len={eval_len:>4}: FAILED — {e}")

    extrap_results[pe_name] = row
    del model; torch.cuda.empty_cache()


# ── Run all 4 PE experiments ────────────────────────────────────────
for pe in ['sinusoidal', 'rope', 'alibi', 'relative']:
    train_and_extrapolate(pe)


# ── Print extrapolation table ───────────────────────────────────────
print("\n" + "═"*58)
print("PART 3 — EXTRAPOLATION TABLE")
print("All models trained on seq_len=512, evaluated on longer seqs")
print("─"*58)
print(f"{'PE Type':<14} {'PPL@512':>10} {'PPL@1024':>10} {'PPL@2048':>10}")
print("─"*58)

for pe_name, row in extrap_results.items():
    def fmt(k):
        v = row.get(k)
        return f"{v:.2f}" if v is not None else " N/A"
    print(f"{pe_name:<14} {fmt('ppl_512'):>10}"
          f" {fmt('ppl_1024'):>10} {fmt('ppl_2048'):>10}")

print("═"*58)
print("\nKey: Lower = better. A small increase from 512→1024→2048 means")
print("     the PE generalises well beyond its training length.")
print_results_table()


═══════════════════════════════════════════════════════
  PE Experiment: SINUSOIDAL
═══════════════════════════════════════════════════════
  Epoch 1 | step 100/294 | loss 9.3778
  Epoch 1 | step 200/294 | loss 8.4180
  ── Epoch 1 | val_ppl@512=1784.85 | time=104s

  Epoch 2 | step 100/294 | loss 7.4258
  Epoch 2 | step 200/294 | loss 7.4308
  ── Epoch 2 | val_ppl@512=1549.41 | time=105s

  Epoch 3 | step 100/294 | loss 7.2899
  Epoch 3 | step 200/294 | loss 7.2674
  ── Epoch 3 | val_ppl@512=1360.05 | time=105s

  Epoch 4 | step 100/294 | loss 7.1854
  Epoch 4 | step 200/294 | loss 7.1699
  ── Epoch 4 | val_ppl@512=1187.37 | time=105s

  Epoch 5 | step 100/294 | loss 7.0197
  Epoch 5 | step 200/294 | loss 6.9874
  ── Epoch 5 | val_ppl@512=1036.64 | time=106s

  Epoch 6 | step 100/294 | loss 6.8176
  Epoch 6 | step 200/294 | loss 6.7800
  ── Epoch 6 | val_ppl@512=873.24 | time=105s

  Epoch 7 | step 100/294 | loss 6.5849
  Epoch 7 | step 200/294 | loss 6.5587
  ── Epoch 7 | val_ppl@512

In [ ]:
# ── Reconstruct all_results from saved experiment outputs ──────────
# Runtime disconnected and lost variables. Rebuilding from known results.
# No retraining needed.

all_results = {
    "Baseline": {
        "name": "Baseline", "attention_type": "standard",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 635.99, "final_val_ppl": 635.99,
        "avg_epoch_time_sec": 104, "peak_gpu_mem_mb": 8013,
        "val_tok_per_sec": 64247,
        "val_perplexities": [635.99],
    },
    "Attn: Sliding Window": {
        "name": "Attn: Sliding Window", "attention_type": "sliding_window",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 547.15, "final_val_ppl": 547.15,
        "avg_epoch_time_sec": 107, "peak_gpu_mem_mb": 8016,
        "val_tok_per_sec": 62936,
        "val_perplexities": [547.15],
    },
    "Attn: Linear": {
        "name": "Attn: Linear", "attention_type": "linear",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 601.43, "final_val_ppl": 601.43,
        "avg_epoch_time_sec": 154, "peak_gpu_mem_mb": 9725,
        "val_tok_per_sec": 47955,
        "val_perplexities": [601.43],
    },
    "Attn: GQA": {
        "name": "Attn: GQA", "attention_type": "gqa",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 15759104, "best_val_ppl": 599.80, "final_val_ppl": 599.80,
        "avg_epoch_time_sec": 104, "peak_gpu_mem_mb": 8010,
        "val_tok_per_sec": 65268,
        "val_perplexities": [599.80],
    },
    "PE: sinusoidal": {
        "name": "PE: sinusoidal", "attention_type": "standard",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 605.67, "final_val_ppl": 605.67,
        "avg_epoch_time_sec": 105, "peak_gpu_mem_mb": 9912,
        "val_tok_per_sec": 64242,
        "val_perplexities": [605.67],
    },
    "PE: rope": {
        "name": "PE: rope", "attention_type": "standard",
        "pe_type": "rope", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 205.44, "final_val_ppl": 205.44,
        "avg_epoch_time_sec": 108, "peak_gpu_mem_mb": 9918,
        "val_tok_per_sec": 62599,
        "val_perplexities": [205.44],
    },
    "PE: alibi": {
        "name": "PE: alibi", "attention_type": "standard",
        "pe_type": "alibi", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 206.73, "final_val_ppl": 206.73,
        "avg_epoch_time_sec": 104, "peak_gpu_mem_mb": 10181,
        "val_tok_per_sec": 64558,
        "val_perplexities": [206.73],
    },
    "PE: relative": {
        "name": "PE: relative", "attention_type": "standard",
        "pe_type": "relative", "conv_type": "None", "seq_len": 512,
        "n_params": 16029440, "best_val_ppl": 273.32, "final_val_ppl": 273.32,
        "avg_epoch_time_sec": 107, "peak_gpu_mem_mb": 9921,
        "val_tok_per_sec": 63207,
        "val_perplexities": [273.32],
    },
}

extrap_results = {
    "sinusoidal": {"pe_type": "sinusoidal",
                   "ppl_512": 605.67, "ppl_1024": 766.28,  "ppl_2048": 2516.35},
    "rope":       {"pe_type": "rope",
                   "ppl_512": 205.44, "ppl_1024": 219.14,  "ppl_2048": 258.15},
    "alibi":      {"pe_type": "alibi",
                   "ppl_512": 206.73, "ppl_1024": 204.99,  "ppl_2048": 204.18},
    "relative":   {"pe_type": "relative",
                   "ppl_512": 273.32, "ppl_1024": 280.82,  "ppl_2048": 293.33},
}

print("all_results reconstructed:")
for k, v in all_results.items():
    print(f"  {k:<28}  best_ppl={v['best_val_ppl']}")

print(f"\nextrap_results reconstructed: {list(extrap_results.keys())}")
print("\nReady for Cell 9 ✓")

all_results reconstructed:
  Baseline                      best_ppl=635.99
  Attn: Sliding Window          best_ppl=547.15
  Attn: Linear                  best_ppl=601.43
  Attn: GQA                     best_ppl=599.8
  PE: sinusoidal                best_ppl=605.67
  PE: rope                      best_ppl=205.44
  PE: alibi                     best_ppl=206.73
  PE: relative                  best_ppl=273.32

extrap_results reconstructed: ['sinusoidal', 'rope', 'alibi', 'relative']

Ready for Cell 9 ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FIXED Cell 9 — Conv Hybrids with CAUSAL convolutions
# Bug fix: symmetric padding let the model see future tokens.
# Fix: pad only on the LEFT so output at position t only sees t and before.
# ═══════════════════════════════════════════════════════════════════

# ── Patch 1: Causal Conv helper ─────────────────────────────────────
class CausalConv1d(nn.Module):
    """
    Causal 1D convolution — output at position t only sees positions <= t.
    We pad (kernel_size - 1) on the LEFT only, then trim the right.
    This shifts the receptive window entirely into the past.
    """
    def __init__(self, channels: int, kernel_size: int, groups: int = 1):
        super().__init__()
        self.pad  = kernel_size - 1
        self.conv = nn.Conv1d(channels, channels, kernel_size,
                              padding=0, groups=groups)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T)
        x_pad = F.pad(x, (self.pad, 0))   # pad LEFT only
        return self.conv(x_pad)            # output length = T  (no future)


# ── Patch 2: Fixed FFN ───────────────────────────────────────────────
class FFN(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg

        if cfg.conv_type == 'gated_conv_ffn':
            self.conv_v = CausalConv1d(cfg.d_model, cfg.conv_kernel_size)
            self.conv_g = CausalConv1d(cfg.d_model, cfg.conv_kernel_size)
            self.proj   = nn.Linear(cfg.d_ff, cfg.d_model)
            # Need to expand channels first for gated conv to have d_ff width
            self.expand_v = nn.Linear(cfg.d_model, cfg.d_ff)
            self.expand_g = nn.Linear(cfg.d_model, cfg.d_ff)
        else:
            self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff)
            self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model)

        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.cfg.conv_type == 'gated_conv_ffn':
            # Expand to d_ff, apply causal conv, gate, project back
            val  = self.expand_v(x).transpose(1, 2)   # (B, d_ff, T)
            gate = self.expand_g(x).transpose(1, 2)
            val  = self.conv_v(val).transpose(1, 2)    # (B, T, d_ff)
            gate = torch.sigmoid(
                   self.conv_g(gate).transpose(1, 2))
            return self.drop(self.proj(val * gate))

        return self.drop(self.fc2(F.gelu(self.fc1(x))))


# ── Patch 3: Fixed TransformerBlock ────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.ffn  = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

        if cfg.conv_type == 'conv_before':
            # Causal depthwise conv — groups=d_model means one filter per channel
            self.conv_pre = nn.Sequential(
                CausalConv1d(cfg.d_model, cfg.conv_kernel_size,
                             groups=cfg.d_model),
                nn.GELU()
            )
        else:
            self.conv_pre = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.conv_pre is not None:
            x = x + self.conv_pre(x.transpose(1, 2)).transpose(1, 2)
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


# ── Quick sanity check ───────────────────────────────────────────────
print("Testing causal property of CausalConv1d...")
conv_test = CausalConv1d(4, 3).to(device)
x_test    = torch.randn(1, 4, 10, device=device)
y_test    = conv_test(x_test)
assert y_test.shape == x_test.shape, "Shape mismatch"
# Output at t=0 should NOT change when we alter t=1 onwards
x_alt           = x_test.clone()
x_alt[:, :, 1:] = torch.randn_like(x_alt[:, :, 1:])
y_alt           = conv_test(x_alt)
assert torch.allclose(y_test[:, :, 0], y_alt[:, :, 0]), "Causality violated!"
print("  Causal check passed ✓ (position 0 output unchanged when future tokens change)")

# ── Find best attention and PE ───────────────────────────────────────
attn_exps = {k: v for k, v in all_results.items()
             if v['attention_type'] != 'standard' or k == 'Baseline'}
pe_exps   = {k: v for k, v in all_results.items()
             if k.startswith('PE:')}

best_attn_name = min(attn_exps, key=lambda k: attn_exps[k]['best_val_ppl'])
best_pe_name   = min(pe_exps,   key=lambda k: pe_exps[k]['best_val_ppl'])
best_attn_type = all_results[best_attn_name]['attention_type']
best_pe_type   = all_results[best_pe_name]['pe_type']

print(f"\nBest attention : {best_attn_name}  ({best_attn_type})")
print(f"Best PE        : {best_pe_name}  ({best_pe_type})")

# ── Conv experiments ─────────────────────────────────────────────────
conv_cfgs = [
    ("Conv: Before-Attn",
     ModelConfig(attention_type=best_attn_type, pe_type=best_pe_type,
                 conv_type='conv_before',    max_seq_len=1024)),
    ("Conv: Gated FFN",
     ModelConfig(attention_type=best_attn_type, pe_type=best_pe_type,
                 conv_type='gated_conv_ffn', max_seq_len=1024)),
]

for name, cfg in conv_cfgs:
    train_model(name=name, cfg=cfg,
                n_epochs=10, seq_len=512, batch_size=16,
                lr=3e-4, verbose=True)

# ── Final table ──────────────────────────────────────────────────────
print("\n\n" + "█"*95)
print(" FINAL RESULTS — ALL EXPERIMENTS")
print("█"*95)
print_results_table()

baseline_ppl = all_results['Baseline']['best_val_ppl']
print(f"\nΔ from baseline (PPL {baseline_ppl:.2f}):")
for k, v in sorted(all_results.items(), key=lambda x: x[1]['best_val_ppl']):
    if k != 'Baseline':
        delta = v['best_val_ppl'] - baseline_ppl
        arrow = "▼ better" if delta < 0 else "▲ worse"
        print(f"  {k:<32} {delta:>+8.2f}  {arrow}")

with open('/content/coreml_results_final.json', 'w') as f:
    json.dump({"experiments": all_results,
               "extrapolation": extrap_results}, f, indent=2)
print("\nSaved: /content/coreml_results_final.json  ✓")
print("Download from Files panel (left sidebar)")

Testing causal property of CausalConv1d...
  Causal check passed ✓ (position 0 output unchanged when future tokens change)

Best attention : Attn: Sliding Window  (sliding_window)
Best PE        : PE: rope  (rope)

════════════════════════════════════════════════════════════
  Experiment: Conv: Before-Attn
  attention=sliding_window  pe=rope  conv=conv_before  seq_len=512
════════════════════════════════════════════════════════════
  Epoch 1 | step  100/294 | loss 10.1517 | ppl 25634.5
  Epoch 1 | step  200/294 | loss 9.1170 | ppl 9108.9

  ── Epoch 1/10 summary ──
     train loss : 8.4467
     val loss   : 6.7856
     val ppl    : 884.98
     epoch time : 101s
     peak mem   : 8081 MB
     val tps    : 61904 tok/s

  Epoch 2 | step  100/294 | loss 6.6163 | ppl 747.1
  Epoch 2 | step  200/294 | loss 6.4748 | ppl 648.6

  ── Epoch 2/10 summary ──
     train loss : 6.3765
     val loss   : 6.1143
     val ppl    : 452.26
     epoch time : 108s
     peak mem   : 8081 MB
     val tps    :

RuntimeError: Given groups=1, weight of size [256, 256, 3], expected input[16, 1024, 514] to have 256 channels, but got 1024 channels instead

In [ ]:
all_results = {
    "Baseline": {
        "name": "Baseline", "attention_type": "standard",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 635.99, "final_val_ppl": 635.99,
        "avg_epoch_time_sec": 104, "peak_gpu_mem_mb": 8013,
        "val_tok_per_sec": 64247, "val_perplexities": [635.99],
    },
    "Attn: Sliding Window": {
        "name": "Attn: Sliding Window", "attention_type": "sliding_window",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 547.15, "final_val_ppl": 547.15,
        "avg_epoch_time_sec": 107, "peak_gpu_mem_mb": 8016,
        "val_tok_per_sec": 62936, "val_perplexities": [547.15],
    },
    "Attn: Linear": {
        "name": "Attn: Linear", "attention_type": "linear",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 601.43, "final_val_ppl": 601.43,
        "avg_epoch_time_sec": 154, "peak_gpu_mem_mb": 9725,
        "val_tok_per_sec": 47955, "val_perplexities": [601.43],
    },
    "Attn: GQA": {
        "name": "Attn: GQA", "attention_type": "gqa",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 15759104, "best_val_ppl": 599.80, "final_val_ppl": 599.80,
        "avg_epoch_time_sec": 104, "peak_gpu_mem_mb": 8010,
        "val_tok_per_sec": 65268, "val_perplexities": [599.80],
    },
    "PE: sinusoidal": {
        "name": "PE: sinusoidal", "attention_type": "standard",
        "pe_type": "sinusoidal", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 605.67, "final_val_ppl": 605.67,
        "avg_epoch_time_sec": 105, "peak_gpu_mem_mb": 9912,
        "val_tok_per_sec": 64242, "val_perplexities": [605.67],
    },
    "PE: rope": {
        "name": "PE: rope", "attention_type": "standard",
        "pe_type": "rope", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 205.44, "final_val_ppl": 205.44,
        "avg_epoch_time_sec": 108, "peak_gpu_mem_mb": 9918,
        "val_tok_per_sec": 62599, "val_perplexities": [205.44],
    },
    "PE: alibi": {
        "name": "PE: alibi", "attention_type": "standard",
        "pe_type": "alibi", "conv_type": "None", "seq_len": 512,
        "n_params": 16021248, "best_val_ppl": 206.73, "final_val_ppl": 206.73,
        "avg_epoch_time_sec": 104, "peak_gpu_mem_mb": 10181,
        "val_tok_per_sec": 64558, "val_perplexities": [206.73],
    },
    "PE: relative": {
        "name": "PE: relative", "attention_type": "standard",
        "pe_type": "relative", "conv_type": "None", "seq_len": 512,
        "n_params": 16029440, "best_val_ppl": 273.32, "final_val_ppl": 273.32,
        "avg_epoch_time_sec": 107, "peak_gpu_mem_mb": 9921,
        "val_tok_per_sec": 63207, "val_perplexities": [273.32],
    },
}

extrap_results = {
    "sinusoidal": {"pe_type": "sinusoidal",
                   "ppl_512": 605.67, "ppl_1024": 766.28,  "ppl_2048": 2516.35},
    "rope":       {"pe_type": "rope",
                   "ppl_512": 205.44, "ppl_1024": 219.14,  "ppl_2048": 258.15},
    "alibi":      {"pe_type": "alibi",
                   "ppl_512": 206.73, "ppl_1024": 204.99,  "ppl_2048": 204.18},
    "relative":   {"pe_type": "relative",
                   "ppl_512": 273.32, "ppl_1024": 280.82,  "ppl_2048": 293.33},
}

print("Reconstructed. Entries:", list(all_results.keys()))

Reconstructed. Entries: ['Baseline', 'Attn: Sliding Window', 'Attn: Linear', 'Attn: GQA', 'PE: sinusoidal', 'PE: rope', 'PE: alibi', 'PE: relative']


In [ ]:
# ── Fix: CausalConv1d for gated FFN must use d_ff channels, not d_model ──

class FFN(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg

        if cfg.conv_type == 'gated_conv_ffn':
            self.expand_v = nn.Linear(cfg.d_model, cfg.d_ff)
            self.expand_g = nn.Linear(cfg.d_model, cfg.d_ff)
            self.conv_v   = CausalConv1d(cfg.d_ff, cfg.conv_kernel_size)  # ← d_ff
            self.conv_g   = CausalConv1d(cfg.d_ff, cfg.conv_kernel_size)  # ← d_ff
            self.proj     = nn.Linear(cfg.d_ff, cfg.d_model)
        else:
            self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff)
            self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model)

        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.cfg.conv_type == 'gated_conv_ffn':
            val  = self.expand_v(x).transpose(1, 2)          # (B, d_ff, T)
            gate = self.expand_g(x).transpose(1, 2)
            val  = self.conv_v(val).transpose(1, 2)           # (B, T, d_ff)
            gate = torch.sigmoid(self.conv_g(gate).transpose(1, 2))
            return self.drop(self.proj(val * gate))
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


# Also re-patch TransformerBlock to pick up the fixed FFN
class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.ffn  = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)
        if cfg.conv_type == 'conv_before':
            self.conv_pre = nn.Sequential(
                CausalConv1d(cfg.d_model, cfg.conv_kernel_size, groups=cfg.d_model),
                nn.GELU()
            )
        else:
            self.conv_pre = None

    def forward(self, x):
        if self.conv_pre is not None:
            x = x + self.conv_pre(x.transpose(1, 2)).transpose(1, 2)
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


# ── Quick shape check before training ────────────────────────────────
cfg_test = ModelConfig(attention_type='sliding_window',
                       pe_type='rope', conv_type='gated_conv_ffn',
                       max_seq_len=1024)
m_test   = TransformerLM(cfg_test).to(device)
dummy    = torch.randint(0, 50257, (2, 64)).to(device)
with torch.no_grad():
    out = m_test(dummy)
assert out.shape == (2, 64, 50257), f"Wrong shape: {out.shape}"
print(f"Shape check passed: {out.shape} ✓")
del m_test; torch.cuda.empty_cache()

# ── Store Conv: Before-Attn result (already completed) ───────────────
all_results["Conv: Before-Attn"] = {
    "name": "Conv: Before-Attn", "attention_type": "sliding_window",
    "pe_type": "rope", "conv_type": "conv_before", "seq_len": 512,
    "n_params": 16025344, "best_val_ppl": 218.71, "final_val_ppl": 218.71,
    "avg_epoch_time_sec": 108, "peak_gpu_mem_mb": 8081,
    "val_tok_per_sec": 61785, "val_perplexities": [218.71],
}

# ── Run only Gated FFN ────────────────────────────────────────────────
cfg_gated = ModelConfig(attention_type='sliding_window',
                        pe_type='rope', conv_type='gated_conv_ffn',
                        max_seq_len=1024)

train_model(name="Conv: Gated FFN", cfg=cfg_gated,
            n_epochs=10, seq_len=512, batch_size=16,
            lr=3e-4, verbose=True)

# ── Final table ───────────────────────────────────────────────────────
print("\n" + "█"*95)
print(" FINAL RESULTS — ALL EXPERIMENTS")
print("█"*95)
print_results_table()

baseline_ppl = all_results['Baseline']['best_val_ppl']
print(f"\nΔ from baseline (PPL {baseline_ppl:.2f}):")
for k, v in sorted(all_results.items(), key=lambda x: x[1]['best_val_ppl']):
    if k != 'Baseline':
        delta = v['best_val_ppl'] - baseline_ppl
        arrow = "▼ better" if delta < 0 else "▲ worse"
        print(f"  {k:<32} {delta:>+8.2f}  {arrow}")

with open('/content/coreml_results_final.json', 'w') as f:
    json.dump({"experiments": all_results,
               "extrapolation": extrap_results}, f, indent=2)
print("\nSaved: /content/coreml_results_final.json ✓")
print("Download from Files panel (left sidebar)")

Shape check passed: torch.Size([2, 64, 50257]) ✓

════════════════════════════════════════════════════════════
  Experiment: Conv: Gated FFN
  attention=sliding_window  pe=rope  conv=gated_conv_ffn  seq_len=512
════════════════════════════════════════════════════════════
  Epoch 1 | step  100/294 | loss 10.0419 | ppl 22969.4
  Epoch 1 | step  200/294 | loss 8.9466 | ppl 7681.8

  ── Epoch 1/10 summary ──
     train loss : 8.2284
     val loss   : 6.4364
     val ppl    : 624.17
     epoch time : 273s
     peak mem   : 9137 MB
     val tps    : 32962 tok/s

  Epoch 2 | step  100/294 | loss 6.2312 | ppl 508.3
  Epoch 2 | step  200/294 | loss 6.1043 | ppl 447.8

  ── Epoch 2/10 summary ──
     train loss : 6.0111
     val loss   : 5.8004
     val ppl    : 330.43
     epoch time : 280s
     peak mem   : 9137 MB
     val tps    : 33007 tok/s

  Epoch 3 | step  100/294 | loss 5.6054 | ppl 271.9
  Epoch 3 | step  200/294 | loss 5.5650 | ppl 261.1

  ── Epoch 3/10 summary ──
     train loss : 